# Develop

## Library Import

In [ ]:
import pandas as pd
import pyspark.sql
from pyspark.ml import Pipeline, PipelineModel

from catboost import CatBoostClassifier

from typing import List

## File Import and model learning (Pipeline)

In [ ]:
def chunk_trainer(files_list: List, file_group: str ='train', chunksize: int = 20000,
                  target_col: str = 'target', model: CatBoostClassifier = None):
    """
    Функция для последовательного обучения модели по чанкам. Принимает следующие аргументы:
    files_list - список с файлами для обучения модели
    file_group - класс файлов для обучения (train или pretrain)
    chunksize - размер чанка (кол-во строк)
    target_col - название целевой фичи
    model - модель
    """
    for file in files_list:
        # chunksize определяет количество строк в одном батче
        reader = pd.read_parquet(f'../datasets/{file_group}/{file}', chunksize=chunksize)

        for i, chunk in enumerate(reader):
            X_batch = chunk.drop(target_col, axis=1)
            y_batch = chunk[target_col]
        
            if i == 0:
                model.fit(X_batch, y_batch)
            else:
                model.fit(X_batch, y_batch, init_model=model)

In [ ]:
train_files = ['train_part_1.parquet', 'train_part_2.parquet', 'train_part_3.parquet']
catboost_model = CatBoostClassifier()
chunk_trainer(files_list=train_files, file_group='train', model=catboost_model)

## Model prediction

In [ ]:
# предсказать на трейне полсе тестсплита для получения внутренних метрик

## Error Analysis

In [ ]:
# Написать кастомную функцию потерь,
# которая будет сильно штрафовать за ложно отрицательные резы

# вывести ее метрику и метрику соревнования

In [ ]:
# Поиск лучших параметров и отбор полезных признаков (shap?)

## Test Predict (подготовка решения)

In [ ]:
test_data = '../datasets/test.parquet'
prediction = catboost_model.predict(test_data)

## Submit Creating (Создание сабмита)

In [ ]:
save_path = '../submits/submit.csv'

def len_error_message(result_df_len: int) -> str:
    return f"Представленный датафрейм имеет иное количество строк: {result_df_len}. Такой сабмит не пройдет. Необходимо 633683 строк"

def create_submit(input_df: List[pd.DataFrame, pyspark.sql.DataFrame]):
    """
    Функция по созданию файла сабмита.
    Принимает input_df - датафрейм с результатом в формате pd.DataFrame или SparkDataFrame
    Сохраняет файл по пути и названию из переменной save_path
    """
    
    if isinstance(input_df, pyspark.sql.DataFrame):
        spark_df = input_df.select('event_id', 'predict')
        row_count = spark_df.count()
        if row_count == 633683:
            pandas_df = spark_df.toPandas()
        else:
            raise ValueError(len_error_message(row_count))
    else:
        pandas_df = input_df.copy()
        row_count = len(pandas_df)
        if row_count != 633683:
            raise ValueError(len_error_message(row_count))
        
    pandas_df.columns = ['event_id', 'predict']
    with open(save_path, 'w') as file:
        pandas_df.to_csv(file, sep=',', index=False)

In [ ]:
create_submit(prediction)